# BFCL tool-calling benchmark: Bonsai 2 9B, original vs counter fine-tuned (Colab A100, ~1–2 h)

Both GGUF files run in PrismML's llama.cpp **on the GPU** (`llama-server`, 8 parallel slots, greedy decoding). Scoring
uses the **official BFCL AST checker** from the gorilla repo (Python categories). On the ground-truth answers it scores
2,295 of 2,311 correct; the 16 exceptions are dataset quirks that affect both models equally.

| run | thinking | categories |
|---|---|---|
| original | off | all 7 below |
| fine-tuned | off | all 7 below |
| original | **on** (the base model's intended mode) | 100 per category |
| fine-tuned | **on** | 100 per category: is thinking still there, and still as good? |

Categories: simple_python (400), multiple (200), parallel (200), parallel_multiple (200), live_simple (258),
live_multiple (300 of 1,053), irrelevance (240, correct = no call).

**Reading the result (fixed now).** The paired comparison original-vs-fine-tuned (both thinking off) uses an exact
McNemar test on the examples where exactly one of them is right. "Fine-tuning improved tool calling" needs a higher
overall score with p < 0.05; a drop on irrelevance (calling a tool when none fits) is reported separately, because
agent data can teach a model to call tools too eagerly.

The original model's runs are cached on Drive from the first evaluation and are not repeated. Set `RUN` in the first cell to choose the fine-tuned model.

Settings: **A100** → Run all. Results go to `My Drive/bonsai_counter/bfcl/`.

## Setup (~20 min: building PrismML's llama.cpp with CUDA)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys, json, time
W = '/content/drive/MyDrive/bonsai_counter'
OUT = f'{W}/bfcl'; os.makedirs(OUT, exist_ok=True)
RUN = 'bonsai2_9b_xlam_fc_t32'          # or 'bonsai2_9b_tooluse_codeagent_t32' (the first, agent-data run)
TAG = 'ft' if RUN == 'bonsai2_9b_tooluse_codeagent_t32' else 'fc'   # result files: {TAG}_nothink.json, {TAG}_think.json
FT = f'{W}/runs/{RUN}/{RUN}.gguf'
assert os.path.exists(FT), f'fine-tuned GGUF not found at {FT}'
assert subprocess.run('nvidia-smi', shell=True).returncode == 0, 'No GPU: Runtime > Change runtime type > A100 GPU, then Run all again'
!pip install -q -U huggingface_hub requests
if not os.path.exists('/content/llama.cpp'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'prism', 'https://github.com/PrismML-Eng/llama.cpp', '/content/llama.cpp'], check=True)
if not os.path.exists('/content/gorilla'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ShishirPatil/gorilla', '/content/gorilla'], check=True)
t0 = time.time()
r = subprocess.run('cmake -S /content/llama.cpp -B /content/llama.cpp/build-cuda -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON '
                   '-DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /content/cmake_cuda.log 2>&1 && '
                   'cmake --build /content/llama.cpp/build-cuda -j --target llama-server > /content/build_cuda.log 2>&1', shell=True)
log = '/content/build_cuda.log' if os.path.exists('/content/build_cuda.log') else '/content/cmake_cuda.log'
print(open(log).read()[-2500:] if r.returncode else f'built llama-server with CUDA in {(time.time() - t0) / 60:.0f} min')
assert r.returncode == 0

In [ ]:
%%writefile /content/bfcl_lite.py
"""BFCL (Berkeley Function Calling Leaderboard) evaluation against an OpenAI-compatible llama-server, scored with the
official BFCL AST checker (loaded from the gorilla repo with its Java/JS and model-registry imports stubbed out;
Python categories only).

Categories: simple_python, multiple, parallel, parallel_multiple, live_simple, live_multiple (AST check) and
irrelevance (correct = no function call)."""
import json, math, os, random, re, enum

AST_CATS = {'simple_python': 'simple', 'multiple': 'multiple', 'parallel': 'parallel',
            'parallel_multiple': 'parallel_multiple', 'live_simple': 'simple', 'live_multiple': 'multiple'}
ALL_CATS = list(AST_CATS) + ['irrelevance']


def load_checker(bfcl_root):
    """Exec the official ast_checker.py with its bfcl_eval imports replaced by stubs; returns ast_checker()."""
    src = open(os.path.join(bfcl_root, 'bfcl_eval/eval_checker/ast_eval/ast_checker.py')).read()
    src = re.sub(r'^from bfcl_eval[^\n]*\([^)]*\)\n', '', src, flags=re.M)
    src = re.sub(r'^from bfcl_eval[^\n]*\n', '', src, flags=re.M)
    assert 'from bfcl_eval' not in src, 'unexpected import layout in ast_checker.py'

    class Language(enum.Enum):
        PYTHON = 'python'; JAVA = 'java'; JAVASCRIPT = 'javascript'

    def _no(*a, **k):
        raise NotImplementedError('Java/JS categories are not evaluated here')
    ns = dict(Language=Language, MODEL_CONFIG_MAPPING={}, JAVA_TYPE_CONVERSION={}, JS_TYPE_CONVERSION={},
              java_type_converter=_no, js_type_converter=_no)
    exec(compile(src, 'ast_checker.py', 'exec'), ns)
    ns['convert_func_name'] = lambda name, model_name: name     # we map names back to the dotted originals ourselves
    return lambda funcs, output, answer, cat: ns['ast_checker'](funcs, output, answer, Language.PYTHON, AST_CATS[cat], 'local')


def load_category(bfcl_root, cat, n=None, seed=0):
    d = os.path.join(bfcl_root, 'bfcl_eval/data')
    rows = [json.loads(l) for l in open(os.path.join(d, f'BFCL_v4_{cat}.json')) if l.strip()]
    if cat in AST_CATS:
        ans = {r['id']: r['ground_truth'] for r in (json.loads(l) for l in open(os.path.join(d, 'possible_answer', f'BFCL_v4_{cat}.json')) if l.strip())}
        rows = [dict(r, answer=ans[r['id']]) for r in rows if r['id'] in ans]
    if n and len(rows) > n:
        rows = random.Random(seed).sample(rows, n)
    return rows


TYPE_MAP = {'dict': 'object', 'float': 'number', 'tuple': 'array', 'any': 'string', 'integer': 'integer',
            'string': 'string', 'boolean': 'boolean', 'array': 'array', 'object': 'object', 'number': 'number'}


def to_json_schema(p):
    """BFCL parameter schemas use Python-ish types ('dict', 'float', 'tuple', 'any'); convert to JSON Schema."""
    if not isinstance(p, dict):
        return p
    out = {}
    for k, v in p.items():
        if k == 'type' and isinstance(v, str):
            out[k] = TYPE_MAP.get(v, 'string')
        elif k == 'properties' and isinstance(v, dict):
            out[k] = {pk: to_json_schema(pv) for pk, pv in v.items()}
        elif k == 'items':
            out[k] = to_json_schema(v)
        else:
            out[k] = v
    return out


def safe_name(name):
    return re.sub(r'[^a-zA-Z0-9_-]', '_', name)[:64]


def tools_for(row):
    """OpenAI tool list + map from the safe name sent to the model back to BFCL's original (often dotted) name."""
    tools, back = [], {}
    for f in row['function']:
        s = safe_name(f['name']); back[s] = f['name']
        tools.append({'type': 'function', 'function': {'name': s, 'description': f.get('description', ''),
                                                       'parameters': to_json_schema(f.get('parameters', {'type': 'dict', 'properties': {}}))}})
    return tools, back


def coerce(v):
    if isinstance(v, str):
        t = v.strip()
        try:
            return json.loads(t)
        except Exception:
            return v
    return v


def parse_calls(message, back):
    """Tool calls from a chat-completion message: the server's parsed tool_calls if present, else <tool_call> blocks in
    the text (JSON {"name", "arguments"} or Qwen XML <function=..><parameter=..>). Returns BFCL's output format:
    [{original_name: {param: value}}]."""
    calls = []
    for tc in (message.get('tool_calls') or []):
        fn = tc.get('function', {})
        args = fn.get('arguments', {})
        if isinstance(args, str):
            try:
                args = json.loads(args) if args.strip() else {}
            except Exception:
                args = {}
        calls.append((fn.get('name', ''), args))
    if not calls:
        text = message.get('content') or ''
        text = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
        for blk in re.findall(r'<tool_call>(.*?)(?:</tool_call>|$)', text, flags=re.S):
            blk = blk.strip()
            m = re.search(r'<function=([^>\s]+)>(.*?)(?:</function>|$)', blk, flags=re.S)
            if m:
                params = {p: coerce(val.strip()) for p, val in re.findall(r'<parameter=([^>\s]+)>(.*?)(?:</parameter>|(?=<parameter=)|$)', m.group(2), flags=re.S)}
                calls.append((m.group(1), params))
                continue
            try:
                j = json.loads(blk)
                calls.append((j.get('name', ''), coerce(j.get('arguments', j.get('parameters', {}))) or {}))
            except Exception:
                pass
    out = []
    for name, args in calls:
        if not isinstance(args, dict):
            args = {}
        out.append({back.get(name, name): dict(args)})            # server-parsed arguments are already typed JSON
    return out


def score(checker, row, cat, calls):
    if cat == 'irrelevance':
        return len(calls) == 0
    if not calls:
        return False
    try:
        return bool(checker(row['function'], calls, row['answer'], cat)['valid'])
    except Exception:
        return False


def mcnemar_p(b, c):
    """Exact two-sided McNemar test on the discordant pairs (b: only A right, c: only B right)."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * p)

## The original model, the checker, and a server helper

In [ ]:
sys.path.insert(0, '/content')
import importlib, requests, concurrent.futures as cf
import bfcl_lite as B; importlib.reload(B)
from huggingface_hub import HfApi, hf_hub_download
REPO = 'prism-ml/Ternary-Bonsai-2-9B-gguf'
cand = [f for f in HfApi().list_repo_files(REPO) if f.endswith('.gguf') and 'PQ2_0' in f and 'MTP' not in f]
ORIG = hf_hub_download(REPO, cand[0], local_dir='/content/bonsai')
ROOT = '/content/gorilla/berkeley-function-call-leaderboard'
CHECK = B.load_checker(ROOT)
N = {'simple_python': 400, 'multiple': 200, 'parallel': 200, 'parallel_multiple': 200, 'live_simple': 258, 'live_multiple': 300, 'irrelevance': 240}
DATA = {c: B.load_category(ROOT, c, N[c], seed=0) for c in B.ALL_CATS}
print({c: len(v) for c, v in DATA.items()})

SERVER = '/content/llama.cpp/build-cuda/bin/llama-server'
import socket
def free_port():
    # Colab's own Jupyter server listens on 8080, so let the OS give us an unused port
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0)); return s.getsockname()[1]
PORT = None

def start(gguf, slots=8, ctx_per_slot=8192):
    global PORT
    subprocess.run(['pkill', '-f', 'llama-server'], check=False)    # leftovers from an interrupted cell
    time.sleep(2)
    PORT = port = free_port()
    p = subprocess.Popen([SERVER, '-m', gguf, '-ngl', '99', '-c', str(slots * ctx_per_slot), '-np', str(slots), '--jinja',
                          '--port', str(port), '--host', '127.0.0.1'], stdout=open(f'/content/server_{port}.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(300):
        try:
            if requests.get(f'http://127.0.0.1:{port}/health', timeout=2).status_code == 200:
                print(f'llama-server up on port {port}'); return p
        except Exception: pass
        if p.poll() is not None: break      # server exited: show its log instead of waiting 10 min
        time.sleep(2)
    print(open(f'/content/server_{port}.log').read()[-3000:]); raise RuntimeError('server did not start')

def ask(row, think):
    tools, back = B.tools_for(row)
    msgs = row['question'][0] if isinstance(row['question'][0], list) else row['question']
    body = dict(messages=msgs, tools=tools, temperature=0.0, max_tokens=4096 if think else 1024,
                chat_template_kwargs={'enable_thinking': bool(think)})
    for attempt in range(3):
        try:
            r = requests.post(f'http://127.0.0.1:{PORT}/v1/chat/completions', json=body, timeout=900).json()
            msg = r['choices'][0]['message']
            return msg, B.parse_calls(msg, back)
        except Exception as e:
            err = e; time.sleep(3)
    return {'error': str(err)}, []

def run_model(tag, gguf, think, n_per_cat=None, slots=8):
    path = f'{OUT}/{tag}.json'
    res = json.load(open(path)) if os.path.exists(path) else {}
    todo = [(c, r) for c in B.ALL_CATS for r in DATA[c][:n_per_cat or None] if r['id'] not in res]
    if not todo: print(tag, 'already done'); return res
    p = start(gguf, slots=slots); t0 = time.time()
    try:
        with cf.ThreadPoolExecutor(slots) as ex:
            futs = {ex.submit(ask, r, think): (c, r) for c, r in todo}
            for i, f in enumerate(cf.as_completed(futs)):
                c, r = futs[f]; msg, calls = f.result()
                res[r['id']] = dict(cat=c, ok=B.score(CHECK, r, c, calls), calls=calls, text=(msg.get('content') or '')[-600:],
                              think_chars=len(msg.get('reasoning_content') or ''))
                if (i + 1) % 200 == 0:
                    json.dump(res, open(path, 'w')); print(f'  {tag}: {i + 1}/{len(todo)} in {(time.time() - t0) / 60:.1f} min', flush=True)
    finally:
        p.terminate(); p.wait()
    json.dump(res, open(path, 'w'))
    print(f'{tag}: {len(todo)} examples in {(time.time() - t0) / 60:.1f} min')
    return res

# smoke test: 3 examples, both models
for tag, g in (('smoke_orig', ORIG), ('smoke_ft', FT)):
    p = start(g, slots=1)
    try:
        for c in ('simple_python', 'parallel', 'irrelevance'):
            msg, calls = ask(DATA[c][0], think=False)
            print(f'{tag} | {c} | parsed {calls} | ok {B.score(CHECK, DATA[c][0], c, calls)} | raw: {repr((msg.get("content") or "")[:160])}')
    finally:
        p.terminate(); p.wait()

## Run (~30–60 min): original and fine-tuned with thinking off, then the original with thinking on (100 per category)

In [ ]:
R = {}
R['orig_nothink'] = run_model('orig_nothink', ORIG, think=False)        # cached on Drive after the first time
R['orig_think'] = run_model('orig_think', ORIG, think=True, n_per_cat=100)
R['ft_nothink'] = run_model(f'{TAG}_nothink', FT, think=False)
R['ft_think'] = run_model(f'{TAG}_think', FT, think=True, n_per_cat=100)

## Results

In [ ]:
def table(A, F, title):
    print(f"\n{title}\n{'category':20s} {'original':>14s} {'fine-tuned':>10s} {'diff':>7s} {'only-orig/only-ft':>18s} {'McNemar p':>10s}")
    out = {}
    groups = [(c, {c}) for c in B_CATS] + [('six call categories', set(B_CATS) - {'irrelevance'}), ('OVERALL', set(B_CATS))]
    for name, cs in groups:
        ids = [k for k, x in A.items() if x['cat'] in cs and k in F]
        if not ids: continue
        a = 100 * sum(A[k]['ok'] for k in ids) / len(ids); b = 100 * sum(F[k]['ok'] for k in ids) / len(ids)
        oo = sum(A[k]['ok'] and not F[k]['ok'] for k in ids); of = sum(F[k]['ok'] and not A[k]['ok'] for k in ids)
        p = Bm.mcnemar_p(oo, of)
        out[name] = dict(n=len(ids), orig=a, ft=b, only_orig=oo, only_ft=of, p=p)
        print(f"{name:20s} {a:9.1f} (n={len(ids):4d}) {b:9.1f} {b - a:+7.1f} {oo:8d} / {of:<8d} {p:10.3g}")
    return out
Bm, B_CATS = B, B.ALL_CATS
summary = {'thinking_off': table(R['orig_nothink'], R['ft_nothink'], f'THINKING OFF: original vs {RUN}'),
           'thinking_on': table(R['orig_think'], R['ft_think'], f'THINKING ON (100 per category): original vs {RUN}')}
for name in ('orig_think', 'ft_think'):
    tc = [x['think_chars'] for x in R[name].values() if 'think_chars' in x]
    if not tc: print(f'{name}: thinking length not recorded (cached from the first evaluation)'); continue
    print(f"{name}: {sum(t > 0 for t in tc)} of {len(tc)} replies contain thinking, mean {sum(tc) / max(len(tc), 1):.0f} chars of it")
json.dump(summary, open(f'{OUT}/summary_{TAG}.json', 'w'), indent=1)
# a few examples where the two models disagree
dis = [k for k in R['ft_nothink'] if k in R['orig_nothink'] and R['ft_nothink'][k]['ok'] != R['orig_nothink'][k]['ok']]
for k in dis[:6]:
    print(f"\n--- {k}: original {'OK' if R['orig_nothink'][k]['ok'] else 'wrong'} {R['orig_nothink'][k]['calls']}\n    fine-tuned {'OK' if R['ft_nothink'][k]['ok'] else 'wrong'} {R['ft_nothink'][k]['calls']}")
print('\nsaved', f'{OUT}/summary_{TAG}.json')